In [1]:
# IMPORT THƯ VIỆN
import os
import cv2
import random
import shutil
import numpy as np
import albumentations as A
from pathlib import Path
 
print("Import hoàn tất.")

Import hoàn tất.


In [2]:
# CẤU HÌNH ĐƯỜNG DẪN VÀ THAM SỐ
 
DATASET_PATH = "/kaggle/input/datasets/dunnguynvn/blood-dataset-dnv/blood_dataset"
CLEAN_PATH   = "/kaggle/working/dataset_clean"
RESIZED_PATH = "/kaggle/working/dataset_resized"
SPLIT_PATH   = "/kaggle/working/dataset_split"
 
CLASSES = [
    "BA", "BNE", "EO", "ERB", "IG", "LY", "MMY",
    "MO", "MY", "MYELOBLAST", "PLATELET", "PMY", "SNE"
]
 
IMG_SIZE = 224
 
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
 
# Tăng target lên 1000 để giảm mất cân bằng tốt hơn
# IG: 151 → 1000 | PMY: ~592 → 1000
IG_TARGET  = 1000
PMY_TARGET = 1000
 
# Số ảnh tối thiểu cố định cho val và test mỗi lớp.
# Với lớp IG chỉ có 151 ảnh gốc, 10% = 15 ảnh quá ít để
# đánh giá tin cậy. Dùng max(10%, 30) đảm bảo tối thiểu
# 30 ảnh trong val và test bất kể tỉ lệ phần trăm.
MIN_VAL_TEST = 30
 
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
 
print("Cấu hình hoàn tất.")
print(f"  Số lớp        : {len(CLASSES)}")
print(f"  Kích thước    : {IMG_SIZE}x{IMG_SIZE} pixels")
print(f"  Tỉ lệ chia    : train={TRAIN_RATIO} / val={VAL_RATIO} / test={TEST_RATIO}")
print(f"  Target IG     : {IG_TARGET} ảnh")
print(f"  Target PMY    : {PMY_TARGET} ảnh")
print(f"  Min val/test  : {MIN_VAL_TEST} ảnh/lớp")

Cấu hình hoàn tất.
  Số lớp        : 13
  Kích thước    : 224x224 pixels
  Tỉ lệ chia    : train=0.8 / val=0.1 / test=0.1
  Target IG     : 1000 ảnh
  Target PMY    : 1000 ảnh
  Min val/test  : 30 ảnh/lớp


In [3]:
# KIỂM TRA VÀ LOẠI BỎ ẢNH CORRUPT
# Ảnh bị coi là corrupt khi cv2.imread() trả về None.
# Ảnh hợp lệ được copy sang CLEAN_PATH để các bước tiếp
# theo không bị ảnh hưởng bởi ảnh lỗi.
 
for cls in CLASSES:
    Path(CLEAN_PATH, cls).mkdir(parents=True, exist_ok=True)
 
removed = 0
kept    = 0
 
for cls in CLASSES:
    src_dir = Path(DATASET_PATH) / cls
    dst_dir = Path(CLEAN_PATH) / cls
 
    for img_path in src_dir.glob("*.*"):
        img = cv2.imread(str(img_path))
        if img is None:
            removed += 1
        else:
            shutil.copy2(str(img_path), str(dst_dir / img_path.name))
            kept += 1
 
print("Kết quả kiểm tra ảnh corrupt:")
print(f"  Đã loại bỏ : {removed} ảnh")
print(f"  Giữ lại    : {kept} ảnh hợp lệ")
print(f"\n  {'Lớp':<15} {'Số ảnh':>8}")
print(f"  {'-'*25}")
for cls in CLASSES:
    count = len(list(Path(CLEAN_PATH, cls).glob("*.*")))
    print(f"  {cls:<15} {count:>8}")

Kết quả kiểm tra ảnh corrupt:
  Đã loại bỏ : 1426 ảnh
  Giữ lại    : 16598 ảnh hợp lệ

  Lớp               Số ảnh
  -------------------------
  BA                  1099
  BNE                 1469
  EO                  2926
  ERB                 1507
  IG                   151
  LY                  1176
  MMY                  991
  MO                  1372
  MY                  1103
  MYELOBLAST          1000
  PLATELET            2172
  PMY                  404
  SNE                 1228


In [4]:
# CHUẨN HÓA KÍCH THƯỚC ẢNH (RESIZE 224x224)
# ResNet50 và Swin Transformer pretrained trên ImageNet với
# kích thước 224x224. INTER_AREA được dùng khi downscaling
# để giảm aliasing (răng cưa) tốt hơn INTER_LINEAR.
 
for cls in CLASSES:
    Path(RESIZED_PATH, cls).mkdir(parents=True, exist_ok=True)
 
for cls in CLASSES:
    src_dir = Path(CLEAN_PATH) / cls
    dst_dir = Path(RESIZED_PATH) / cls
 
    for img_path in src_dir.glob("*.*"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE),
                                 interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(dst_dir / img_path.name), img_resized)
 
print("Resize hoàn tất. Kiểm tra kết quả:")
print(f"\n  {'Lớp':<15} {'Số ảnh':>8} {'Kích thước':>12}")
print(f"  {'-'*38}")
 
for cls in CLASSES:
    files = list(Path(RESIZED_PATH, cls).glob("*.*"))
    if files:
        sample = cv2.imread(str(files[0]))
        size   = f"{sample.shape[1]}x{sample.shape[0]}"
    else:
        size = "N/A"
    print(f"  {cls:<15} {len(files):>8} {size:>12}")

Resize hoàn tất. Kiểm tra kết quả:

  Lớp               Số ảnh   Kích thước
  --------------------------------------
  BA                  1099      224x224
  BNE                 1469      224x224
  EO                  2926      224x224
  ERB                 1507      224x224
  IG                   151      224x224
  LY                  1176      224x224
  MMY                  991      224x224
  MO                  1372      224x224
  MY                  1103      224x224
  MYELOBLAST          1000      224x224
  PLATELET            2172      224x224
  PMY                  404      224x224
  SNE                 1228      224x224


In [5]:
# ĐỊNH NGHĨA PIPELINE TĂNG CƯỜNG DỮ LIỆU
# Các phép biến đổi phù hợp với ảnh tế bào máu:
#   - HorizontalFlip / VerticalFlip  : tế bào không có hướng cố định
#   - RandomRotate90 / Rotate        : bất biến với góc xoay
#   - ColorJitter                    : mô phỏng biến thiên màu nhuộm
#   - GaussNoise                     : tăng tính tổng quát hóa
#   - ElasticTransform               : mô phỏng biến dạng nhẹ của tế bào
#
# GaussNoise: Albumentations >= 2.0 đổi tham số từ
#   var_limit → std_range để thống nhất với định nghĩa thống kê.
#   std_range=(0.02, 0.08) tương đương var_limit=(5, 20).
 
aug_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=30, p=0.7),
    A.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05,
        p=0.8
    ),
    A.GaussNoise(std_range=(0.02, 0.08), p=0.3),
    A.ElasticTransform(alpha=1, sigma=10, p=0.3),
])
 

In [6]:
# TĂNG CƯỜNG DỮ LIỆU OFFLINE CHO LỚP IG
# IG chỉ có 151 ảnh sau khi loại corrupt → mục tiêu 1000 ảnh.
# Kiểm tra và xóa ảnh aug cũ trước khi tạo mới để tránh
# ghi đè khi chạy lại notebook nhiều lần.
 
ig_dir = Path(RESIZED_PATH) / "IG"
 
ig_existing_aug = list(ig_dir.glob("ig_aug_*.jpg"))
if ig_existing_aug:
    print(f"Phát hiện {len(ig_existing_aug)} ảnh aug cũ của IG → xóa để chạy lại sạch.")
    for f in ig_existing_aug:
        f.unlink()
 
ig_images = list(ig_dir.glob("*.*"))
ig_needed = IG_TARGET - len(ig_images)
 
print(f"\nLớp IG hiện có : {len(ig_images)} ảnh gốc")
print(f"Mục tiêu       : {IG_TARGET} ảnh")
print(f"Cần tạo thêm   : {ig_needed} ảnh")
 
for i in range(ig_needed):
    src_path      = random.choice(ig_images)
    img           = cv2.imread(str(src_path))
    img_rgb       = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    augmented     = aug_pipeline(image=img_rgb)["image"]
    augmented_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(ig_dir / f"ig_aug_{i:04d}.jpg"), augmented_bgr)
 
ig_total = len(list(ig_dir.glob("*.*")))
print(f"Lớp IG sau tăng cường: {ig_total} ảnh")


Lớp IG hiện có : 151 ảnh gốc
Mục tiêu       : 1000 ảnh
Cần tạo thêm   : 849 ảnh
Lớp IG sau tăng cường: 1000 ảnh


In [7]:
# TĂNG CƯỜNG DỮ LIỆU OFFLINE CHO LỚP PMY
# PMY có ~592 ảnh → mục tiêu 1000 ảnh.
# Tương tự IG: kiểm tra và xóa aug cũ trước khi tạo mới.
 
pmy_dir = Path(RESIZED_PATH) / "PMY"
 
pmy_existing_aug = list(pmy_dir.glob("pmy_aug_*.jpg"))
if pmy_existing_aug:
    print(f"Phát hiện {len(pmy_existing_aug)} ảnh aug cũ của PMY → xóa để chạy lại sạch.")
    for f in pmy_existing_aug:
        f.unlink()
 
pmy_images = list(pmy_dir.glob("*.*"))
pmy_needed = PMY_TARGET - len(pmy_images)
 
print(f"\nLớp PMY hiện có : {len(pmy_images)} ảnh gốc")
print(f"Mục tiêu        : {PMY_TARGET} ảnh")
print(f"Cần tạo thêm    : {pmy_needed} ảnh")
 
for i in range(pmy_needed):
    src_path      = random.choice(pmy_images)
    img           = cv2.imread(str(src_path))
    img_rgb       = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    augmented     = aug_pipeline(image=img_rgb)["image"]
    augmented_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(pmy_dir / f"pmy_aug_{i:04d}.jpg"), augmented_bgr)
 
pmy_total = len(list(pmy_dir.glob("*.*")))
print(f"Lớp PMY sau tăng cường: {pmy_total} ảnh")


Lớp PMY hiện có : 404 ảnh gốc
Mục tiêu        : 1000 ảnh
Cần tạo thêm    : 596 ảnh
Lớp PMY sau tăng cường: 1000 ảnh


In [8]:
# KIỂM TRA KÊNH MÀU RGB
# ResNet50 và Swin Transformer yêu cầu ảnh đầu vào có đúng
# 3 kênh màu (RGB). Bước này xác nhận toàn bộ ảnh đúng định dạng.
 
wrong_channel = []
 
for cls in CLASSES:
    for img_path in Path(RESIZED_PATH, cls).glob("*.*"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        if len(img.shape) != 3 or img.shape[2] != 3:
            wrong_channel.append((cls, str(img_path), img.shape))
 
if wrong_channel:
    print(f"Cảnh báo: Có {len(wrong_channel)} ảnh sai kênh màu:")
    for cls, path, shape in wrong_channel[:5]:
        print(f"  [{cls}] shape={shape} -> {path}")
else:
    print("Kết quả: Toàn bộ ảnh đã là 3 kênh RGB, không cần xử lý thêm.")

Kết quả: Toàn bộ ảnh đã là 3 kênh RGB, không cần xử lý thêm.


In [9]:
# PHÂN CHIA TẬP TRAIN / VAL / TEST (80/10/10)
# Nguyên tắc áp dụng:
#   1. Stratified split: Mỗi lớp được chia theo đúng tỉ lệ trên
#      cả 3 tập → phân bố lớp đồng đều.
#   2. Ảnh tăng cường offline chỉ vào train: val và test chỉ
#      chứa ảnh gốc để đánh giá khách quan.
#   3. random.seed(42): Kết quả phân chia reproducible.
#   4. Cố định MIN_VAL_TEST=30 ảnh cho val/test mỗi lớp:
#      Dùng max(10%, 30) đảm bảo tối thiểu 30 ảnh trong val
#      và test bất kể tỉ lệ phần trăm — quan trọng với IG.
 
for split in ["train", "val", "test"]:
    for cls in CLASSES:
        Path(SPLIT_PATH, split, cls).mkdir(parents=True, exist_ok=True)
 
random.seed(RANDOM_SEED)
 
for cls in CLASSES:
    all_images       = list(Path(RESIZED_PATH, cls).glob("*.*"))
    original_images  = [p for p in all_images if "_aug_" not in p.name]
    augmented_images = [p for p in all_images if "_aug_" in p.name]
 
    random.shuffle(original_images)
    n = len(original_images)
 
    # Cố định tối thiểu MIN_VAL_TEST ảnh cho val và test
    n_val   = max(int(n * VAL_RATIO),  MIN_VAL_TEST)
    n_test  = max(int(n * TEST_RATIO), MIN_VAL_TEST)
    n_train = n - n_val - n_test
 
    # Trường hợp đặc biệt: ảnh gốc quá ít, không đủ để chia
    if n_train < 0:
        n_val   = n // 3
        n_test  = n // 3
        n_train = n - n_val - n_test
        print(f"  {cls}: ảnh gốc quá ít ({n} ảnh), "
              f"buộc chia train={n_train} / val={n_val} / test={n_test}")
 
    split_map = {
        "train": original_images[:n_train],
        "val"  : original_images[n_train : n_train + n_val],
        "test" : original_images[n_train + n_val :]
    }
 
    # Ảnh tăng cường chỉ vào train
    split_map["train"] += augmented_images
 
    for split_name, files in split_map.items():
        for f in files:
            shutil.copy2(
                str(f),
                str(Path(SPLIT_PATH, split_name, cls, f.name))
            )
 
print("Phân chia tập dữ liệu hoàn tất.")

Phân chia tập dữ liệu hoàn tất.


In [10]:
# CELL 10: KIỂM TRA KẾT QUẢ CUỐI CÙNG
# Hiển thị thống kê đầy đủ:
#   - Số ảnh từng lớp trong train/val/test
#   - Tỉ lệ thực tế sau phân chia
#   - Cảnh báo lớp còn ít ảnh val/test
#   - Mức độ mất cân bằng còn lại trong train
 
print("=" * 62)
print("THỐNG KÊ BỘ DỮ LIỆU SAU TIỀN XỬ LÝ")
print("=" * 62)
print(f"  {'Lớp':<15} {'Train':>8} {'Val':>8} {'Test':>8} {'Tổng':>8}")
print(f"  {'-'*55}")
 
total_train  = total_val = total_test = 0
warnings     = []
train_counts = []
 
for cls in CLASSES:
    n_train = len(list(Path(SPLIT_PATH, "train", cls).glob("*.*")))
    n_val   = len(list(Path(SPLIT_PATH, "val",   cls).glob("*.*")))
    n_test  = len(list(Path(SPLIT_PATH, "test",  cls).glob("*.*")))
    n_total = n_train + n_val + n_test
 
    total_train  += n_train
    total_val    += n_val
    total_test   += n_test
    train_counts.append(n_train)
 
    if n_val < MIN_VAL_TEST:
        warnings.append(f"    {cls}: val chỉ có {n_val} ảnh (< {MIN_VAL_TEST})")
    if n_test < MIN_VAL_TEST:
        warnings.append(f"    {cls}: test chỉ có {n_test} ảnh (< {MIN_VAL_TEST})")
 
    print(f"  {cls:<15} {n_train:>8} {n_val:>8} {n_test:>8} {n_total:>8}")
 
grand_total = total_train + total_val + total_test
 
print(f"  {'-'*55}")
print(f"  {'TOTAL':<15} {total_train:>8} {total_val:>8} {total_test:>8} {grand_total:>8}")
print("=" * 62)
 
# Tỉ lệ thực tế
print(f"\nTỉ lệ thực tế:")
print(f"  Train : {total_train/grand_total*100:.1f}%")
print(f"  Val   : {total_val/grand_total*100:.1f}%")
print(f"  Test  : {total_test/grand_total*100:.1f}%")
 
# Cảnh báo lớp ít ảnh
if warnings:
    print(f"\nCảnh báo phân phối:")
    for w in warnings:
        print(w)
else:
    print(f"\n Tất cả lớp đều có ít nhất {MIN_VAL_TEST} ảnh trong val và test.")
 
# Mức độ mất cân bằng còn lại trong train
imbalance_ratio = max(train_counts) / min(train_counts)
print(f"\nMức độ mất cân bằng trong train:")
print(f"  Lớp nhiều nhất : {max(train_counts)} ảnh")
print(f"  Lớp ít nhất    : {min(train_counts)} ảnh")
print(f"  Tỉ lệ          : {imbalance_ratio:.1f}x")
 
if imbalance_ratio > 5:
    print(f"    Vẫn còn mất cân bằng đáng kể "
          f"→ dùng class_weight và WeightedSampler khi train.")
else:
    print(f"   Mức độ mất cân bằng chấp nhận được.")
 
print(f"\nĐường dẫn dataset đã xử lý : {SPLIT_PATH}")
print(f"\nCấu trúc thư mục:")
print(f"  dataset_split/")
print(f"    train/  ({total_train} ảnh, bao gồm ảnh tăng cường IG và PMY)")
print(f"    val/    ({total_val} ảnh, chỉ ảnh gốc, tối thiểu {MIN_VAL_TEST} ảnh/lớp)")
print(f"    test/   ({total_test} ảnh, chỉ ảnh gốc, tối thiểu {MIN_VAL_TEST} ảnh/lớp)")
 

THỐNG KÊ BỘ DỮ LIỆU SAU TIỀN XỬ LÝ
  Lớp                Train      Val     Test     Tổng
  -------------------------------------------------------
  BA                   881      109      109     1099
  BNE                 1177      146      146     1469
  EO                  2342      292      292     2926
  ERB                 1207      150      150     1507
  IG                   940       30       30     1000
  LY                   942      117      117     1176
  MMY                  793       99       99      991
  MO                  1098      137      137     1372
  MY                   883      110      110     1103
  MYELOBLAST           800      100      100     1000
  PLATELET            1738      217      217     2172
  PMY                  920       40       40     1000
  SNE                  984      122      122     1228
  -------------------------------------------------------
  TOTAL              14705     1669     1669    18043

Tỉ lệ thực tế:
  Train : 81.5%
  Val  

In [11]:
# NÉN VÀ LƯU OUTPUT
# Nén dataset_split thành zip để notebook train có thể
# add vào làm input dataset trên Kaggle.
 
print("Đang nén dataset_split, vui lòng chờ...")
 
shutil.make_archive(
    "/kaggle/working/dataset_split",
    "zip",
    "/kaggle/working",
    "dataset_split"
)
 
print("Hoàn tất! Danh sách file output:")
for f in sorted(Path("/kaggle/working").glob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name:<30} {size_mb:.1f} MB")

Đang nén dataset_split, vui lòng chờ...
Hoàn tất! Danh sách file output:
  __notebook__.ipynb             0.0 MB
  dataset_split.zip              262.8 MB
